# 10. 실습실 zero-shot 베이스라인

담당: 손이 비는 팀원 누구나 실행할 수 있습니다.
목표: 학습 없이 첫 정확도 숫자와 첫 제출 파일을 가장 빠르게 확보하는 것입니다.

이 노트북은 재실행해도 안전합니다. `python -m vqa.predict` 가 이미 끝난 ID를 건너뛰기 때문에,
중간에 커널이 죽거나 셀을 멈추더라도 같은 셀을 다시 실행하면 이어서 진행됩니다.

## 통과 기준(PASS)
1. val 예측이 끝나고 `evaluate_run` 이 정확도를 출력합니다. 무작위 추측은 25퍼센트이므로 그보다
   확실히 높아야 의미가 있습니다.
2. 전체 test 예측 전에 측정한 문항당 시간으로 6714문항 예상 소요 시간을 확인했습니다.
3. 제출 파일이 `scripts/check_submission.py` 검사를 통과합니다.

## 넘겨줄 내용
정확도, 문항당 시간, 제출 파일 경로와 해시를 수정에게 전달합니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

## 1. 파라미터와 모델 선택

| MODEL_CHOICE | 모델 | 필요한 환경 | 비고 |
|---|---|---|---|
| `qwen25vl_3b` | Qwen2.5-VL-3B-Instruct | transformers 4.x | 오늘 바로 돌아갑니다. 첫 숫자를 뽑는 용도로 가장 안전합니다. |
| `qwen35_4b` | Qwen3.5-4B | transformers 5.x | 비양자화 fallback 후보입니다. |
| `qwen35_9b` | Qwen3.5-9B (NF4) | transformers 5.x | 실습실 주력 모델이며 가장 느립니다. |

여러 대의 PC로 test를 나눠서 처리하려면 `SHARD` 를 `"1/3"`, `"2/3"` 처럼 서로 다르게 지정합니다.
한 대에서 전부 처리할 때는 빈 문자열로 둡니다.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."
DATA_ROOT = r"."
RUN_ROOT = r"./artifacts/runs"
LOCAL_SCRATCH = r"./artifacts/scratch"
MODEL_ROOT = r""

MODEL_CHOICE = "qwen25vl_3b"   # "qwen25vl_3b" | "qwen35_4b" | "qwen35_9b"
RUN_ID = "lab_zeroshot_v1"     # 모델을 바꾸면 RUN_ID도 함께 바꾸십시오.

VAL_IDS_FILE = "fold_csv_files/val_fold0.csv"    # 팀 공식 검증 분할 (fold 0, 1,344문항)
SPLIT_FILE = "artifacts/data/split_fold0.json"   # 팀 공식 학습 분할 (train_fold1234, 5,358문항)
VAL_OUT_NAME = "fold0_base"
TEST_CSV = "test.csv"
N_TEST_ITEMS = 6714            # 전체 test 문항 수

SHARD = ""                     # 예: "1/3". 한 대에서 전부 돌리면 빈 문자열입니다.
N_SHARDS_TOTAL = 3             # 병합 셀에서 사용할 전체 shard 개수
TIMING_LIMIT = 20              # 속도 측정에 사용할 문항 수

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False, hint=True):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다.

    quiet가 True이면 명령의 출력도, 뒤에 붙는 "[rc=...]" 줄도 찍지 않습니다. 상태를 조회하는
    호출은 실패가 정상적인 결과입니다. 예를 들어 `vqa.jobs status` 는 죽은 job에 대해 종료 코드
    1을 돌려줍니다. 그 줄을 그대로 찍으면 셀 출력 맨 앞에 "[rc=1]"이 남아서, 실제 실패로 읽히는
    오해를 부릅니다.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    if not quiet:
        print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        if hint:
            print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다. name 에 / 가 있으면 저장소 루트 기준 경로로 봅니다."""
    path = Path(name) if "/" in name else Path("scripts") / name
    return run([PY, str(path)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

In [ ]:
CONFIGS = {
    "qwen25vl_3b": "configs/lab_qwen25vl_3b.yaml",
    "qwen35_4b": "configs/lab_qwen35_4b.yaml",
    "qwen35_9b": "configs/lab_qwen35_9b.yaml",
}
CONFIG = CONFIGS[MODEL_CHOICE]
RUN_DIR = RUN_ROOT_P / RUN_ID
PRED_DIR = RUN_DIR / "preds"
print("config :", CONFIG, "(존재:", Path(CONFIG).exists(), ")")
print("run dir:", RUN_DIR)
if not Path(VAL_IDS_FILE).exists():
    print("경고: val ID 파일이 아직 없습니다. 수빈이 01번 노트북을 끝낸 뒤에 진행하십시오.")

## 2. validation zero-shot 예측

adapter를 붙이지 않은 base 모델로 고정 validation을 추론합니다. 시간이 오래 걸리면 셀을 멈췄다가
다시 실행해도 됩니다. 이미 처리한 ID는 건너뜁니다.

In [ ]:
vqa("predict",
    "--config", CONFIG,
    "--run-id", RUN_ID,
    "--csv", "train.csv",
    "--ids-file", VAL_IDS_FILE,
    "--adapter", "none",
    "--out-name", VAL_OUT_NAME)

## 3. validation 정확도

정확도가 25퍼센트 근처라면 프롬프트나 단일 토큰 매핑에 문제가 있을 가능성이 큽니다.
그 경우에는 preflight 로그의 단일 토큰 검사 결과부터 확인하십시오.

In [ ]:
Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
script("evaluate_run.py",
       "--pred", str(PRED_DIR / (VAL_OUT_NAME + ".csv")),
       "--gold", str(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv"),
       "--val-ids", VAL_IDS_FILE,
       "--json-out", "artifacts/eval/%s_%s.json" % (RUN_ID, VAL_OUT_NAME),
       "--wrong-out", "artifacts/eval/%s_%s_wrong.csv" % (RUN_ID, VAL_OUT_NAME),
       "--note", "zero-shot %s" % MODEL_CHOICE)

## 4. 전체 test에 들어가기 전에 시간부터 재기

적은 수의 문항으로 실제 속도를 측정하고, 6714문항 기준 예상 시간을 계산합니다. 이 숫자를 보고
몇 대의 PC에 나눌지 결정하십시오. 예상 시간이 남은 실습 시간보다 길면 더 작은 모델로 내려가거나
PC를 더 투입해야 합니다.

In [ ]:
timing = vqa("predict",
             "--config", CONFIG,
             "--run-id", RUN_ID,
             "--csv", TEST_CSV,
             "--adapter", "none",
             "--limit", TIMING_LIMIT,
             "--out-name", "timing_probe",
             check=False)

sec_per_item = timing["seconds"] / max(TIMING_LIMIT, 1)
print()
print("측정 결과: %.2f초/문항 (모델 로딩 시간이 포함된 값이라 보수적입니다)" % sec_per_item)
print()
print("전체 %d문항 예상 소요 시간" % N_TEST_ITEMS)
print("  PC 대수 | 예상 시간")
for n_pc in (1, 2, 3, 4, 5):
    hours = sec_per_item * N_TEST_ITEMS / 3600.0 / n_pc
    print("  %6d대 | %5.2f시간" % (n_pc, hours))
print()
print("모델 로딩이 한 번뿐인 긴 실행에서는 실제 속도가 이보다 빠릅니다.")
print("전체 실행을 시작한 뒤 진행률 표시로 실제 속도를 다시 확인하십시오.")

## 5. 전체 test 예측

위 예상 시간을 확인한 뒤에 실행하십시오. `SHARD` 를 지정하면 이 PC가 맡은 부분만 처리합니다.
중간에 멈춰도 같은 셀을 다시 실행하면 이어서 진행됩니다.

In [ ]:
args = ["--config", CONFIG, "--run-id", RUN_ID, "--csv", TEST_CSV,
        "--adapter", "none", "--out-name", "test_base"]
if SHARD:
    args += ["--shard", SHARD]
    print("이 PC가 맡은 구간:", SHARD)
else:
    print("전체 구간을 이 PC에서 처리합니다.")

vqa("predict", *args)

## 6. shard 병합

여러 PC로 나눠서 돌렸다면, 각 PC의 `preds/test_base.shard{i}of{n}.csv` 파일을 한 폴더에 모은 뒤에
이 셀을 실행합니다. 각 ID가 정확히 한 번씩만 들어 있는지 병합기가 검사합니다.
한 대에서 전부 처리했다면 이 셀은 건너뛰어도 됩니다.

In [ ]:
shard_files = sorted(PRED_DIR.glob("test_base.shard*of*.csv"))
print("모인 shard 파일 %d개" % len(shard_files))
for f in shard_files:
    print(" -", f.name)

if shard_files:
    script("merge_shards.py",
           "--shards", *[str(f) for f in shard_files],
           "--expect-ids", str(Path(os.environ["VQA_DATA_ROOT"]) / TEST_CSV),
           "--seconds-per-item", "%.4f" % sec_per_item,
           "--out", str(PRED_DIR / "test_base.csv"))
else:
    print("병합할 shard가 없습니다. 단일 PC 실행이라면 정상입니다.")

## 7. 제출 파일 생성과 검사

`check_submission.py` 가 통과해야 실제로 제출할 수 있습니다. 파싱에 실패한 행을 임의로 `a` 로
채우지 않도록 되어 있으므로, 결측이 보고되면 해당 ID를 다시 추론해야 합니다.

In [ ]:
SUB_PATH = "artifacts/submission/%s_base.csv" % RUN_ID
Path("artifacts/submission").mkdir(parents=True, exist_ok=True)

vqa("submit",
    "--pred", str(PRED_DIR / "test_base.csv"),
    "--sample", "sample_submission.csv",
    "--out", SUB_PATH)

script("check_submission.py",
       "--submission", SUB_PATH,
       "--sample", "sample_submission.csv")

print()
print("===== 제출 후보 공유 =====")
print("run_id     : %s (%s)" % (RUN_ID, MODEL_CHOICE))
print("제출 파일  : %s" % SUB_PATH)
print("파일 sha256: %s" % sha256_of(SUB_PATH))
print("문항당 시간: %.2f초" % sec_per_item)
print("다음 담당  : 수정이 30번 노트북에서 실험 기록에 추가합니다.")
print("=========================")